# Same Story, Different Headlines

In this notebook you'll compare how four very different outlets — the
New York Times, the Wall Street Journal, Fox News, and China's state agency
Xinhua — headline the *same* US-China events. Then you'll train a model to guess
which outlet wrote a headline from its wording alone... and try to fool it.

**A note on the data:** the headlines in this notebook are written *in the style of*
each outlet for teaching purposes — they're not scraped real headlines. The framing
patterns they carry are the real subject.

**No coding experience needed.** Almost everything is pre-written. In marked spots
you'll edit a line of real Python — a string or a list — and watch what changes.

---

### ⚠️ Step zero — this is YOUR copy

This notebook is your personal copy — it has your name on the file. Your edits save
automatically. If it ever gets hopelessly tangled, **Ctrl+Z / Cmd+Z inside a cell
fixes almost everything** — and if it doesn't, email Heagen for a fresh copy.

### How this notebook works

- Cells run **top to bottom**: click a cell, press **Shift+Enter** (or ▶).
- **▶️ JUST RUN** cells: run, don't edit.
- **✏️ YOUR TURN** cells: change the line marked `# <-- EDIT`.
- Under every ✏️ cell is a **"What you should see"** box for checking yourself.

**Time:** built for ~4 hours. Setup ~30 min · Beat 1 ~60 min · Beat 2 ~60 min ·
Beat 3 ~60 min · Stretch ~30 min.

## Setup

**▶️ JUST RUN**: installs two libraries and loads the headlines. Takes 1–2 minutes;
lots of text will scroll by. Wait for the ✅.

In [ ]:
# ▶️ JUST RUN — installs and imports (takes 1-2 minutes)
%pip install -q sentence-transformers vaderSentiment

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option("display.max_colwidth", 100)

DATA_URL = "https://raw.githubusercontent.com/YOUR-ORG/trade-war-lab-notebooks/main/data/headlines.csv"
headlines = pd.read_csv(DATA_URL)
print(f"Loaded {len(headlines)} headlines: {headlines.outlet.nunique()} outlets x "
      f"{headlines.event.nunique()} events. ✅")
headlines.head()

> **What you should see:** `Loaded 72 headlines: 4 outlets x 6 events. ✅` and a
> table with columns `event`, `outlet`, `headline`.

Which events are covered? **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — the six events
list(headlines["event"].unique())

> **What you should see:** six event names, from `'2018 tariff announcement'` to
> `'Spy balloon'`.

**Do this before anything else:** pick one event and read all 12 of its headlines.
**▶️ JUST RUN** (then change the number 0–5 if you like — this one's freebie-editable):

In [ ]:
# ▶️ JUST RUN — read one event's headlines side by side
event_to_read = list(headlines["event"].unique())[2]
headlines[headlines.event == event_to_read][["outlet", "headline"]]

> **What you should see:** 12 headlines about the Phase One deal. Same signing
> ceremony — read down the column and watch it become four different stories.

Your eye already noticed the framing differences. The rest of this notebook asks:
can we *measure* what your eye just did? One more setup cell. **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — helper tools
vader = SentimentIntensityAnalyzer()
headlines["sentiment"] = headlines["headline"].map(
    lambda t: vader.polarity_scores(t)["compound"])

def sentiment_by_outlet(event):
    """Average headline sentiment per outlet for one event. -1 = very negative, +1 = very positive."""
    d = headlines[headlines.event == event]
    if len(d) == 0:
        print(f"(No event called '{event}' — options: {list(headlines.event.unique())})")
        return
    means = d.groupby("outlet")["sentiment"].mean().reindex(["NYT", "WSJ", "Fox", "Xinhua"])
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["#2ca02c" if v >= 0 else "#d62728" for v in means]
    ax.bar(means.index, means.values, color=colors)
    ax.axhline(0, color="gray")
    ax.set_ylim(-1, 1); ax.set_ylabel("avg. sentiment score")
    ax.set_title(f'Headline sentiment: "{event}"')
    plt.tight_layout(); plt.show()
    for outlet, v in means.items():
        print(f"{outlet}: {v:+.2f}")

def score_one(text):
    """VADER sentiment of any single sentence you type."""
    s = vader.polarity_scores(text)["compound"]
    print(f'"{text}"  →  sentiment {s:+.2f}')

print("Sentiment tools ready ✅ — on to Beat 1!")

---
## Beat 1 — Measuring the spin ✏️

We'll score every headline with a **sentiment analyzer** (called VADER), a tool
that rates text from −1 (very negative) to +1 (very positive) by adding up scores
of individual words.

**✏️ YOUR TURN:** run as-is first, then swap in other events from the list above.

In [ ]:
event = "Phase One deal"   # <-- EDIT (copy any event name from the list above)

sentiment_by_outlet(event)

> **What you should see** (with the starting event): a bar chart and printed scores
> roughly `NYT: -0.21  WSJ: -0.11  Fox: +0.05  Xinhua: +0.48`.
> **If you get a red error:** the event name needs quotes and must match the list
> exactly, including capitalization.

### Things to try

1. Look at that Phase One pattern. The two outlets closest to the two *governments*
   that signed the deal (Fox → Trump administration, Xinhua → Beijing) both spin
   positive. The two papers of record are both skeptical: "hard issues unresolved,"
   "enforcement questions loom." Scroll up to the raw headlines and find the exact
   words driving each score.
2. Try `"Spy balloon"`. Now Xinhua is the *most negative* — but read its headlines:
   negative at *the U.S.* ("overreaction," "poisons ties"). Sentiment tells you the
   emotional charge, but not the **target**. What would you need beyond a
   positive/negative score to capture that?
3. Now the trap: try `"Hong Kong security law"`. Every outlet scores *positive*,
   including headlines about a crackdown. Why? VADER just adds word scores, and
   "security," "stability," "freedom," "safety" all count positive, even in
   "Beijing Crushes Hong Kong Freedom." Test it yourself in a scratch cell
   (Insert → Code cell): `score_one("Beijing crushes Hong Kong freedom")`.
   **A tool that counts words can't read.** Hold that thought. The next beat brings
   a model that can do better.

---
## Beat 2 — Guess the outlet

A harder question: can a model tell **who wrote it** from
wording alone?

We'll use a language model that turns each headline into numbers capturing its
*meaning and style*, then train a classifier to map those numbers to outlets. It has
never seen the 22 test headlines we hold back.

**▶️ JUST RUN** (takes ~1 minute, since it's downloading the language model):

In [ ]:
# ▶️ JUST RUN — build and train the outlet classifier
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X = embedder.encode(list(headlines["headline"]), show_progress_bar=True)
y = headlines["outlet"].values

X_train, X_test, y_train, y_test, head_train, head_test = train_test_split(
    X, y, headlines, test_size=0.3, random_state=42, stratify=y)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Trained on {len(X_train)} headlines.")
print(f"On {len(X_test)} headlines it has NEVER seen, it guesses the outlet right {accuracy:.0%} of the time.")
print(f"(Random guessing among 4 outlets would get 25%.)")

def guess_outlet(text):
    """The model's guess — with its confidence in each of the four outlets."""
    probs = model.predict_proba(embedder.encode([text]))[0]
    order = np.argsort(probs)[::-1]
    print(f'"{text}"')
    for i in order:
        bar = "█" * int(probs[i] * 30)
        print(f"  {model.classes_[i]:<7} {probs[i]:>5.0%} {bar}")

def quiz():
    """Show a random test headline. Guess the outlet yourself, then reveal."""
    row = head_test.sample(1).iloc[0]
    guess = model.predict(embedder.encode([row.headline]))[0]
    print(f'HEADLINE: "{row.headline}"')
    print("Think: who wrote this? ...")
    print(f"Model's guess: {guess}   |   Actual: {row.outlet}   |   "
          f"{'model was RIGHT ✅' if guess == row.outlet else 'model was WRONG ❌'}")

def battery(my_headlines):
    """Run a list of your own headlines through the classifier."""
    if isinstance(my_headlines, str):
        my_headlines = [my_headlines]
    rows = [(h[:65], model.predict(embedder.encode([h]))[0]) for h in my_headlines]
    return pd.DataFrame(rows, columns=["your headline", "model's guess"])

print("\nGame on — run the quiz cell below!")

> **What you should see:** an accuracy line — likely somewhere in the **60–85%**
> range. Against a 25% random baseline, that means headline *style alone* carries a
> strong outlet signature.

Now play. **▶️ JUST RUN this cell over and over**: each run deals a new headline.
Guess the outlet in your head before reading the reveal. Keep your own score against
the model's.

In [ ]:
# ▶️ JUST RUN — again and again. New headline every run.
quiz()

> **What you should see:** a headline, the model's guess, the actual outlet, and
> whether the model was right.

After ten rounds or so: what tells are *you* using? "Communist" screams one outlet;
"win-win" another; "investors" a third. Now check whether the model found the same
tells.

**✏️ YOUR TURN:** feed it headlines and watch its confidence spread across outlets.

In [ ]:
my_headline = "Markets steady as trade negotiators meet in Beijing"   # <-- EDIT

guess_outlet(my_headline)

> **What you should see:** your headline followed by four confidence bars, largest
> on top — for the starting headline, WSJ should lead comfortably.
> **If you get a red error:** keep the quotation marks around your headline.

### Things to try

1. Write the most WSJ headline you can. Then mutate it word by word toward Fox
   ("Markets" → "Patriots"? "steady" → "furious"?) and watch the bars shift. Find
   the single word swap that moves the most probability.
2. Write a headline about something with no US-China content at all: a bake sale, a
   football game. The model *still* picks an outlet. It has to. It only knows four
   answers. What does that tell you about trusting a classifier outside the world it
   was trained on?

---
## Beat 3 — Fool it ✏️

You've probed it; now break it. Your mission: write headlines that carry one
outlet's *worldview* in another outlet's *clothing*. A pro-Beijing message in WSJ
market-speak. A Fox-style alarm written with NYT restraint.

**✏️ YOUR TURN:** replace these with your own attempts, then run.

In [ ]:
my_headlines = [
    "Investors cheer as tariffs teach Beijing a lesson",          # <-- EDIT
    "Analysts question whether balloon response was warranted",    # <-- EDIT
    "Win-win cooperation lifts stock futures higher",              # <-- EDIT: add more, comma-separated
]

battery(my_headlines)

> **What you should see:** a table pairing each of your headlines with the model's
> guess.
> **If you get a red error:** each headline needs its own quotes, commas between
> them, all inside the square brackets.

### Things to try

1. Score yourself: a successful fool is a headline where you *intended* one outlet's
   worldview but the model names another. What's your hit rate after ten attempts?
2. Reverse-engineer your wins. When you fooled it, which words did the work? You're
   mapping the model's learned features by hand. This is a real research technique
   (it's called probing, and people publish papers doing it).
3. The deeper question: when the model says "Fox," is it detecting *ideology*, or
   just vocabulary statistics that usually travel with ideology? Your best fooling
   headlines are evidence. Which side do they support?

---
## Stretch goal (optional) — style or subject?

Here's a sharper version of that last question. Maybe the model isn't learning
outlet *style* at all. Maybe it just memorized which words go with which events.
One way to find out: train it with an entire event **hidden**, then test it only on
that unseen event. If it still guesses outlets well, it learned style that
transfers. If it collapses toward 25%, it was leaning on event vocabulary.

**✏️ YOUR TURN:**

In [ ]:
hidden_event = "Spy balloon"   # <-- EDIT (any event from the list)

train_rows = headlines[headlines.event != hidden_event]
test_rows  = headlines[headlines.event == hidden_event]
m2 = LogisticRegression(max_iter=2000)
m2.fit(embedder.encode(list(train_rows.headline)), train_rows.outlet)
acc = m2.score(embedder.encode(list(test_rows.headline)), test_rows.outlet)
print(f"Trained with '{hidden_event}' completely hidden.")
print(f"Accuracy on that unseen event: {acc:.0%}  (random = 25%)")

> **What you should see:** an accuracy number for the hidden event. Above ~50% means
> real style transfer; near 25% means the model was memorizing event words.

Rotate through all six events. Is transfer even, or are some events easier to
generalize to? Write one sentence on what that pattern suggests.

(You just ran what researchers call a *held-out domain* test, one of the standard
ways to check whether a model learned the thing you meant it to learn.)

---
## What you just did

With a handful of edited strings and lists, you:

- measured how four outlets spin the same events — and caught the two government-adjacent
  outlets both cheerleading the deal their side signed,
- found VADER's blind spot (it can't tell "crushing freedom" from "freedom"),
- trained a classifier that identifies an outlet from wording alone, quizzed
  yourself against it, and mapped its tells,
- attacked it with adversarial headlines, and
- ran a held-out domain test to ask whether it learned style or memorized vocabulary.

**The bridge to Week 2:** everything you did today was *classification* — reading
text and assigning a label, then asking whether the label-assigner can be trusted.
Next week you're on the other side: **you** assign the labels ("is this bill about
China?") that the lab's classifier learns from. After today you know exactly what a
sloppy label does downstream — and why your careful ones matter.

**Before you close:** write down (1) your best fooling headline, and (2) your answer
to the style-vs-vocabulary question, in one sentence. Bring both to the Week 2 kickoff.